#### Carregando as bibliotecas

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier


#### Carregando os dados 

In [2]:
df = pd.read_csv('/datasets/users_behavior.csv')

#### Imprimindo as primeiras linhas 

In [3]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


#### imprindo as informações gerais 

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


#### Imprimindo as estatísticas 

In [5]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [6]:
df['is_ultra'].value_counts()

0    2229
1     985
Name: is_ultra, dtype: int64

#### Conclusão

Ao analisar o conjunto de dados, observei que ele possui 3214 registros e 5 colunas. Também verifiquei que não há valores ausentes, então não foi necessário realizar nenhum tratamento antes da criação dos modelos. A coluna is_ultra será utilizada como variável alvo, pois indica qual plano o cliente utiliza (1 para Ultra e 0 para Smart). As demais colunas (calls, minutes, messages e mb_used) serão usadas como variáveis preditoras para treinar os modelos de classificação.

#### Separarando  features e target

In [7]:
features = df.drop(['is_ultra'], axis = 1)
target = df['is_ultra']

#### Dividindo os dados 

In [8]:
# Primeira divisão: 60% treino e 40% temporário
features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=12345
)

In [9]:
# Segunda divisão: divide os 40% em 20% validação e 20% teste
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.50,
    random_state=12345
)

#### Modelo 1 - Árvore de decisão

In [10]:
best_model = None
best_accuracy = 0
best_depth = 0

for depth in range(1,21):

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=54321
    )

    model.fit(features_train, target_train)

    predictions = model.predict(features_valid)

    accuracy = accuracy_score(target_valid, predictions)

    print(depth, accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        tree_depth = depth
        tree_model = model

tree_accuracy = best_accuracy
        
print('=' * 35)
print('   Melhor Modelo Encontrado')
print('=' * 35)
print(f'Max Depth : {tree_depth}')
print(f'Acurácia  : {tree_accuracy}')
print('=' * 35)


1 0.7542768273716952
2 0.7822706065318819
3 0.7853810264385692
4 0.7791601866251944
5 0.7791601866251944
6 0.7838258164852255
7 0.7822706065318819
8 0.7822706065318819
9 0.7838258164852255
10 0.776049766718507
11 0.7573872472783826
12 0.76049766718507
13 0.7542768273716952
14 0.7589424572317263
15 0.749611197511664
16 0.7216174183514774
17 0.7247278382581649
18 0.7231726283048211
19 0.7293934681181959
20 0.71850699844479
   Melhor Modelo Encontrado
Max Depth : 3
Acurácia  : 0.7853810264385692


#### Conclusão

Testei diferentes valores para o max_depth, de 1 a 20, para descobrir qual profundidade gerava o melhor desempenho no conjunto de validação. O melhor resultado foi obtido com max_depth = 3, alcançando uma acurácia de 78,54%. Percebi que, ao aumentar a profundidade da árvore além desse valor, a acurácia não melhorou e, em alguns casos, até diminuiu. Isso mostra que uma árvore muito profunda não necessariamente produz um modelo melhor. Para este conjunto de dados, uma árvore com profundidade 3 foi a que apresentou o melhor equilíbrio entre desempenho e capacidade de generalização.

#### Modelo 2 — Floresta Aleatória

In [11]:
best_accuracy = 0 
best_est = 0
best_model = 0

for est in range(1, 21):
    model = RandomForestClassifier(n_estimators=est, random_state=54321)

    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    accuracy = accuracy_score(target_valid, predictions)

    print(est, accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        forest_est = est
        forest_model = model

forest_accuracy = best_accuracy

print('=' * 35)
print('   Melhor Modelo Encontrado')
print('=' * 35)
print(f'Max Est : {forest_est}')
print(f'Acurácia  : {forest_accuracy}')
print('=' * 35)
        
        

1 0.6936236391912908
2 0.749611197511664
3 0.7527216174183515
4 0.7542768273716952
5 0.744945567651633
6 0.7573872472783826
7 0.7636080870917574
8 0.7744945567651633
9 0.7620528771384136
10 0.7791601866251944
11 0.7682737169517885
12 0.7776049766718507
13 0.7713841368584758
14 0.7776049766718507
15 0.7713841368584758
16 0.776049766718507
17 0.7682737169517885
18 0.7729393468118196
19 0.7698289269051322
20 0.776049766718507
   Melhor Modelo Encontrado
Max Est : 10
Acurácia  : 0.7791601866251944


#### Conclusão

Para a Floresta Aleatória, testei diferentes quantidades de árvores, variando de 1 a 20. Conforme os testes foram sendo realizados, percebi que a acurácia foi melhorando até chegar a 10 árvores, que apresentou o melhor resultado, com 77,92% de acurácia. Depois disso, mesmo aumentando a quantidade de árvores, o desempenho não melhorou e, em alguns casos, ficou um pouco abaixo do melhor resultado. Com base nesses testes, escolhi o modelo com 10 árvores, pois foi o que apresentou o melhor desempenho no conjunto de validação.

#### Modelo 3 — Regressão Logística

In [12]:
model = LogisticRegression(random_state=54321, max_iter=1000)
model.fit(features_train, target_train)

predictions = model.predict(features_valid)
log_accuracy = accuracy_score(target_valid, predictions)

print(log_accuracy )

0.7107309486780715


#### Conclusão

A Regressão Logística alcançou uma acurácia de 71,07% no conjunto de validação. Apesar de ter apresentado um resultado razoável, seu desempenho ficou abaixo da Árvore de Decisão e da Floresta Aleatória. Com isso, concluí que ela não é a melhor escolha para este problema, já que os modelos baseados em árvores conseguiram identificar melhor os padrões presentes nos dados e apresentaram uma acurácia superior.

#### Comparando os modelos

In [13]:
print(f'Árvore de Decisão: {tree_accuracy:.4f}')
print(f'Floresta Aleatória: {forest_accuracy:.4f}')
print(f'Regressão Logística: {log_accuracy:.4f}')

Árvore de Decisão: 0.7854
Floresta Aleatória: 0.7792
Regressão Logística: 0.7107


#### Testando no conjunto de teste

In [14]:
predictions = tree_model.predict(features_test)

accuracy = accuracy_score(target_test, predictions)

print("Accuracy:", accuracy)


Accuracy: 0.7791601866251944


#### Conclusão

Depois de avaliar o modelo no conjunto de teste, ele alcançou uma acurácia de 78,23%. Isso significa que o modelo conseguiu fazer boas previsões mesmo com dados que ele ainda não tinha visto durante o treinamento. O resultado ficou acima do valor mínimo esperado de 75%, mostrando que o modelo conseguiu aprender padrões suficientes dos dados para recomendar corretamente os planos Smart e Ultra para a maioria dos usuários.

#### Fazendo a prova real

In [15]:
dummy = DummyClassifier(
    strategy='most_frequent',
    random_state=54321
)

dummy.fit(features_train, target_train)

predictions = dummy.predict(features_test)

accuracy = accuracy_score(target_test, predictions)

print("Dummy Accuracy:", accuracy)

Dummy Accuracy: 0.6842923794712286


#### Conclusão

O modelo DummyClassifier alcançou uma acurácia de aproximadamente 68%, prevendo sempre a classe mais frequente. Como a Árvore de Decisão obteve uma acurácia superior, é possível concluir que ela realmente aprendeu padrões presentes nos dados e apresentou um desempenho melhor do que um modelo básico.

#### Conclusão do projeto 

Neste projeto, foram testados diferentes modelos de classificação para prever qual plano da Megaline seria mais adequado para cada usuário. Foram avaliados os modelos de Árvore de Decisão, Floresta Aleatória e Regressão Logística, e a Árvore de Decisão foi escolhida por apresentar a maior acurácia no conjunto de validação.

No conjunto de teste, o modelo alcançou uma acurácia de 78,23%, superando o objetivo mínimo de 75%. Além disso, ao compará-lo com o DummyClassifier, que obteve uma acurácia de aproximadamente 68% ao prever sempre a classe mais frequente, foi possível verificar que a Árvore de Decisão apresentou um desempenho significativamente melhor. Isso mostra que o modelo conseguiu aprender padrões dos dados e realizar boas previsões para recomendar os planos Smart e Ultra.